In [1]:
import json
import pandas as pd
import string
from unidecode import unidecode
import os

In [2]:
JV_REPLACE=True

In [3]:
def jv_replace(text: str):
    text = text.replace('j', 'i')
    text = text.replace('v', 'u')
    return text

In [4]:
data_dir = '../data/final_dataset'
short_ans_files = ['certamen_short_answer.json', 'junior_scholarship_short_answer.json']
short_ans_files = [os.path.join(data_dir, f) for f in short_ans_files]

file_to_data = {}
for file in short_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_short_answer.json 4596
junior_scholarship_short_answer.json 675


In [5]:
file_to_resp = {}
#model_name = 'qwq'
#model_name = 'llama3-turbo'
model_name = 'o3-mini'

model_resp_dir = f'../data/model_responses/{model_name}'

for file in file_to_data.keys():
    base_name = os.path.basename(file)
    file_to_resp[base_name] = {}
    with open(os.path.join(model_resp_dir, base_name), 'r') as f:
        file_to_resp[base_name] = json.load(f)
    print(base_name, len(file_to_resp[base_name]))





certamen_short_answer.json 4596
junior_scholarship_short_answer.json 675


In [6]:
# make dfs
question_dfs = []
for file in file_to_data.keys():
    questions = file_to_data[file]
    question_df = pd.DataFrame(questions)
    question_dfs.append(question_df)

In [7]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    answer_dict = file_to_resp[file]
    answer_col = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']

        model_answer = answer_dict[q_id]
        answer_col.append(model_answer)

    question_df['raw_resp'] = answer_col

In [8]:
def parse_resp(resp_text):
    '''get the answer from the full response. usually on the last line as Answer: answer'''

    lines = resp_text.split('\n')
    # usually in last line
    line = lines[-1]
    line = line.replace('*', '')
    # look for "Answer: A"
    if 'Answer:' in line:
        try:
            answer =line.split('Answer: ')[1].strip()
        except:
            print(line)
        return answer
    else:
        print('error parsing')
        print(resp_text)
        return ''


In [9]:
def score_single_short_answer(correct_answers: list, model_answer: str, answer_language: str, ignore_macrons: bool):
    '''assumes only 1 correct answer'''

    correct_answer = correct_answers[0].lower().strip()
    if not model_answer:
        return 0
    model_answer = model_answer.lower().strip()

    # strip any punctuation 
    correct_answer = correct_answer.translate(str.maketrans('', '', string.punctuation))
    model_answer = model_answer.translate(str.maketrans('', '', string.punctuation))

    # normalize any macrons in both answers 
    if ignore_macrons and answer_language == 'latin':
        correct_answer = unidecode(correct_answer)
        model_answer = unidecode(model_answer)

    if JV_REPLACE and answer_language == 'latin':
        correct_answer = jv_replace(correct_answer)
        model_answer = jv_replace(model_answer)
        

    return int(correct_answer == model_answer)


In [10]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['parsed_resp'] = question_df['raw_resp'].apply(parse_resp)
    question_dfs[i] = question_df
question_dfs[0]

error parsing
I’m sorry, but I don’t see the image or any additional details that would allow me to determine which island is labeled “A.” Could you please provide more context or a description of the map or diagram you’re referring to? That way I can help you identify the island correctly.


,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,multihop,raw_resp,parsed_resp
0,NJCL-Certamen,1996,NJCL-Certamen_1996_1a_0,short_answer,history,unknown,english,latin,What name was given to the large agricultural ...,[],[LATIFUNDIA],False,"These vast estates, resulting from the redistr...",latifundia
1,NJCL-Certamen,1996,NJCL-Certamen_1996_1b_0,short_answer,history,unknown,english,latin,What was the manager or overseer of a latifund...,[],[VILICUS],False,"In ancient Rome, large agricultural estates kn...",vilicus
2,NJCL-Certamen,1996,NJCL-Certamen_1996_4a_0,short_answer,vocabulary,unknown,latin,latin,Quod animal facit mel?,[],[APIS],False,"In Latin, ""Quod animal facit mel?"" translates ...",apis
3,NJCL-Certamen,1996,NJCL-Certamen_1996_6b_0,short_answer,literature,unknown,english,english,Who was Odysseus' long-suffering wife?,[],[Penelope],False,"Odysseus' wife, renowned for her patience and ...",Penelope
4,NJCL-Certamen,1996,NJCL-Certamen_1996_7a_0,short_answer,grammar,unknown,english,latin,"Give the first person singular, present passiv...",[],[VEHAR],False,The verb vehō (“I carry”) has the principal pa...,vehar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591,NJCL-Certamen,2002,NJCL-Certamen_2002_7b_46,short_answer,vocabulary,unknown,english,english,"Of a librarian, a soprano, a goalie, or a ball...",[],[ballerina],False,"To ""vertiginate"" (a playful neologism echoing ...",ballerina
4592,NJCL-Certamen,2002,NJCL-Certamen_2002_3_60,short_answer,mythology,advanced,english,latin,Which of the suitors of Penelope was the ringl...,[],[ANTINOUS],False,"In Homer’s Odyssey, Antinous is portrayed as t...",Antinous
4593,NJCL-Certamen,2002,NJCL-Certamen_2002_12_61,short_answer,history,advanced,english,latin,"In 107 BC, who served the first of his eventua...",[],[Marius],False,"In 107 BC, it was Gaius Marius who served what...",Marius
4594,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_659,short_answer,history,unknown,english,latin,The columna rostrāta was erected in the Forum ...,[],[DUILIUS],True,The columna rostrāta in the Forum Rōmānum was ...,Duilius


In [11]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    scores_macrons = []
    scores_no_macrons = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']
        correct_answers = row['answers']
        model_answer = row['parsed_resp']
        answer_language = row['answer_language']
        scores_macrons.append(score_single_short_answer(correct_answers, model_answer, answer_language, ignore_macrons=False))
        scores_no_macrons.append(score_single_short_answer(correct_answers, model_answer, answer_language, ignore_macrons=True))

    question_df['score'] = scores_macrons
    question_df['score_ignore_macrons'] = scores_no_macrons
question_dfs[0]




,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,multihop,raw_resp,parsed_resp,score,score_ignore_macrons
0,NJCL-Certamen,1996,NJCL-Certamen_1996_1a_0,short_answer,history,unknown,english,latin,What name was given to the large agricultural ...,[],[LATIFUNDIA],False,"These vast estates, resulting from the redistr...",latifundia,1,1
1,NJCL-Certamen,1996,NJCL-Certamen_1996_1b_0,short_answer,history,unknown,english,latin,What was the manager or overseer of a latifund...,[],[VILICUS],False,"In ancient Rome, large agricultural estates kn...",vilicus,1,1
2,NJCL-Certamen,1996,NJCL-Certamen_1996_4a_0,short_answer,vocabulary,unknown,latin,latin,Quod animal facit mel?,[],[APIS],False,"In Latin, ""Quod animal facit mel?"" translates ...",apis,1,1
3,NJCL-Certamen,1996,NJCL-Certamen_1996_6b_0,short_answer,literature,unknown,english,english,Who was Odysseus' long-suffering wife?,[],[Penelope],False,"Odysseus' wife, renowned for her patience and ...",Penelope,1,1
4,NJCL-Certamen,1996,NJCL-Certamen_1996_7a_0,short_answer,grammar,unknown,english,latin,"Give the first person singular, present passiv...",[],[VEHAR],False,The verb vehō (“I carry”) has the principal pa...,vehar,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591,NJCL-Certamen,2002,NJCL-Certamen_2002_7b_46,short_answer,vocabulary,unknown,english,english,"Of a librarian, a soprano, a goalie, or a ball...",[],[ballerina],False,"To ""vertiginate"" (a playful neologism echoing ...",ballerina,1,1
4592,NJCL-Certamen,2002,NJCL-Certamen_2002_3_60,short_answer,mythology,advanced,english,latin,Which of the suitors of Penelope was the ringl...,[],[ANTINOUS],False,"In Homer’s Odyssey, Antinous is portrayed as t...",Antinous,1,1
4593,NJCL-Certamen,2002,NJCL-Certamen_2002_12_61,short_answer,history,advanced,english,latin,"In 107 BC, who served the first of his eventua...",[],[Marius],False,"In 107 BC, it was Gaius Marius who served what...",Marius,1,1
4594,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_659,short_answer,history,unknown,english,latin,The columna rostrāta was erected in the Forum ...,[],[DUILIUS],True,The columna rostrāta in the Forum Rōmānum was ...,Duilius,1,1


In [12]:
# create combined df, with new columns called "source file"
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['source_file'] = [file] * len(question_df)
    question_dfs[i] = question_df

combined_df = pd.concat(question_dfs)
combined_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,multihop,raw_resp,parsed_resp,score,score_ignore_macrons,source_file,question_text
0,NJCL-Certamen,1996,NJCL-Certamen_1996_1a_0,short_answer,history,unknown,english,latin,What name was given to the large agricultural ...,[],[LATIFUNDIA],False,"These vast estates, resulting from the redistr...",latifundia,1,1,certamen_short_answer.json,NaN
1,NJCL-Certamen,1996,NJCL-Certamen_1996_1b_0,short_answer,history,unknown,english,latin,What was the manager or overseer of a latifund...,[],[VILICUS],False,"In ancient Rome, large agricultural estates kn...",vilicus,1,1,certamen_short_answer.json,NaN
2,NJCL-Certamen,1996,NJCL-Certamen_1996_4a_0,short_answer,vocabulary,unknown,latin,latin,Quod animal facit mel?,[],[APIS],False,"In Latin, ""Quod animal facit mel?"" translates ...",apis,1,1,certamen_short_answer.json,NaN
3,NJCL-Certamen,1996,NJCL-Certamen_1996_6b_0,short_answer,literature,unknown,english,english,Who was Odysseus' long-suffering wife?,[],[Penelope],False,"Odysseus' wife, renowned for her patience and ...",Penelope,1,1,certamen_short_answer.json,NaN
4,NJCL-Certamen,1996,NJCL-Certamen_1996_7a_0,short_answer,grammar,unknown,english,latin,"Give the first person singular, present passiv...",[],[VEHAR],False,The verb vehō (“I carry”) has the principal pa...,vehar,1,1,certamen_short_answer.json,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_app.V.1.1,short_answer,vocabulary,unknown,english,english,NaN,[],[Over-robe.],False,"In ancient Rome the toga was a distinctive, se...",Garment,0,0,junior_scholarship_short_answer.json,What is a toga?
671,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_app.V.1.2,short_answer,vocabulary,unknown,english,english,NaN,[],[Under-garment.],False,A tunica was a basic garment in ancient Roman ...,Tunic,0,0,junior_scholarship_short_answer.json,What is a tunica?
672,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_app.V.9.1,short_answer,vocabulary,unknown,english,latin,NaN,[],[*Carceres*],False,One classical Latin term that was used in the ...,meta,0,0,junior_scholarship_short_answer.json,"Give the Latin for ""starting-post."""
673,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_app.V.9.2,short_answer,vocabulary,unknown,english,latin,NaN,[],[*meta*],False,In Latin one might express the idea by taking ...,Mediameta,0,0,junior_scholarship_short_answer.json,"Give the Latin for ""half-way-post."""


In [13]:
accuracy = combined_df['score'].mean()
accuracy

0.5911591728324797

In [14]:
combined_df['score_ignore_macrons'].mean()

0.6448491747296529

In [15]:
# save combined df
save_dir = f'../data/model_responses_parsed/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# drop the columns question, multiple_choice_options, n_required, correctness_logic
save_df = combined_df.drop(columns=['question', 'question_text', 'multiple_choice_options'])
save_dict = save_df.to_dict(orient='records')



with open(os.path.join(save_dir, 'short_answer.json'), 'w') as f:
    json.dump(save_dict, f, indent=4)

In [15]:
def accuracy_by(df, group_cols, score_cols):
    """
    df          : DataFrame
    group_cols  : column or list of columns to group on
    score_cols  : str or list of str with binary-score columns
    """
    if isinstance(score_cols, str):
        score_cols = [score_cols]
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    # reshape wide → long so we can handle any number of *_score columns
    long = (
        df
        .melt(id_vars=group_cols,          # keep the grouping keys
              value_vars=score_cols,       # columns like 'bert_score', 'roberta_score', ...
              var_name="model",            # model = original column name
              value_name="score")          # 0/1 values
    )

    return (
        long
        .groupby(group_cols + ["model"], dropna=False)
        .score
        .agg(accuracy="mean",  n_items="size")
        .reset_index()
    )

In [24]:
def accuracy_by(df, group_cols, score_cols=("score", "score_ignore_macrons")):
    """
    df          : DataFrame
    group_cols  : column or list of columns to group on
    score_cols  : iterable of score column names
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    score_cols = list(score_cols)

    # Build named aggregations for each score column
    named_aggs = {f"accuracy_{col}": (col, "mean") for col in score_cols}
    # Use any existing column for size (all give same count per group)
    any_col = score_cols[0]
    named_aggs["n_items"] = (any_col, "size")

    return (
        df.groupby(group_cols, dropna=False)
          .agg(**named_aggs)
          .reset_index()
    )

In [25]:
acc_difficulty = accuracy_by(combined_df,
                             group_cols="difficulty",
                             #score_cols=model_name+"_score"
                             )
acc_difficulty

,difficulty,accuracy_score,accuracy_score_ignore_macrons,n_items
0,advanced,0.634069,0.684543,951
1,beginner,0.675214,0.726496,117
2,unknown,0.583631,0.629788,4203


In [26]:
acc_lang_combos = accuracy_by(combined_df,
                             group_cols=["question_language", "answer_language"],
                             #score_cols=model_name+'_score'
                             )
acc_lang_combos

,question_language,answer_language,accuracy_score,accuracy_score_ignore_macrons,n_items
0,english,english,0.636713,0.636713,1302
1,english,latin,0.591015,0.648403,3851
2,latin,english,0.888889,0.888889,9
3,latin,latin,0.201835,0.449541,109


In [27]:
acc_content = accuracy_by(combined_df,
                             group_cols="question_content",
                             #score_cols=model_name+'_score'
                             )
acc_content

,question_content,accuracy_score,accuracy_score_ignore_macrons,n_items
0,geography,0.775641,0.782051,156
1,grammar,0.323323,0.448448,999
2,history,0.748879,0.765695,892
3,literary_devices,0.260870,0.304348,46
4,literature,0.595300,0.605744,383
5,mythology,0.743261,0.751348,1484
6,scansion,0.000000,0.277778,18
7,translation,0.375000,0.642857,56
8,vocabulary,0.532741,0.588521,1237


In [27]:
lang_combos = question_df[["question_language", "answer_language"]].value_counts()
content_types = question_df['question_content'].unique()

In [28]:
def add_missing_content_cols(df):
    cols = list(df.columns)
    new_df = df.copy()

    for content_type in content_types:
        if content_type not in cols:
            new_df[content_type] = [0]

    # sort cols
    sorted_types = sorted(list(content_types))
    new_df = new_df[list(sorted_types)]

    return new_df

In [29]:
sorted_types = sorted(list(content_types))
print('lang_combo', ' '.join(sorted_types))

for q_lang, a_lang in lang_combos.index:
    print(f"{q_lang}-{a_lang}", end=' ')
    # df with only this combo of langs 
    lang_df = question_df[(question_df['question_language'] == q_lang) &
                          (question_df['answer_language'] == a_lang)]
    
    # unique content types 
    proportions = lang_df['question_content'].value_counts()
    df = pd.DataFrame([proportions]) # transposes it
    df = add_missing_content_cols(df)
    print(' '.join([str(x) for x in df.iloc[0].values]))


lang_combo geography grammar history literary_devices literature mythology translation vocabulary
english-latin 57 80 69 27 75 82 48 51
english-english 43 6 30 18 25 18 9 47
latin-latin 0 14 1 0 0 0 0 2


In [30]:
acc_lang_content = accuracy_by(question_df,
                             group_cols=["question_language", "answer_language", "question_content"],
                             score_cols=model_name+'_score')
acc_lang_content

,question_language,answer_language,question_content,model,accuracy,n_items
0,english,english,geography,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.441860,43
1,english,english,grammar,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.166667,6
2,english,english,history,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.666667,30
3,english,english,literary_devices,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.222222,18
4,english,english,literature,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.400000,25
5,english,english,mythology,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.666667,18
6,english,english,translation,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.777778,9
7,english,english,vocabulary,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.574468,47
8,english,latin,geography,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.736842,57
9,english,latin,grammar,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score,0.400000,80


In [31]:
# filter df to look at questions 

# english-english, translation questions 
question_df[(question_df['question_language'] == 'english') &
            (question_df['answer_language'] == 'english') &
            (question_df['question_content'] == 'translation')]


,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,meta-llama_Meta-Llama-3-70B-Instruct-Turbo,meta-llama_Meta-Llama-3-70B-Instruct-Turbo_score
572,NJCL-Certamen,2000,NJCL-Certamen_2000_10c_15,short_answer,translation,beginner,english,english,"In hāc sententiā, quid Anglicē significat “Qua...",[],[HOW],How,1
574,NJCL-Certamen,2002,NJCL-Certamen_2002_13b_50,short_answer,translation,advanced,english,english,The seal also contains the year in which the u...,[],[1820],1820,1
582,NJCL-Certamen,1996,NJCL-Certamen_1996_6c_7,short_answer,translation,unknown,english,english,Translate quod into English for this Latin sen...,[],[which],Which,1
587,NJCL-Certamen,2000,NJCL-Certamen_2000_20a_8,short_answer,translation,unknown,english,english,How should one translate the verb form amābō i...,[],[PLEASE],Shall,0
590,NJCL-Certamen,1996,NJCL-Certamen_1996_6a_9,short_answer,translation,unknown,english,english,Translate quod into English for this Latin sen...,[],[which],Which,1
591,NJCL-Certamen,1996,NJCL-Certamen_1996_16b_3,short_answer,translation,unknown,english,english,Translate cum into English for this Latin sent...,[],[although],When,0
594,NJCL-Certamen,1996,NJCL-Certamen_1996_2a_38,short_answer,translation,unknown,english,english,Translate ut into English for this sentence: U...,[],[How],How!,1
600,NJCL-Certamen,1996,NJCL-Certamen_1996_6b_9,short_answer,translation,unknown,english,english,Translate quod into English for this Latin sen...,[],[because],because,1
601,NJCL-Certamen,1996,NJCL-Certamen_1996_2c_38,short_answer,translation,unknown,english,english,Translate ut into English for this sentence: T...,[],[That],that,1
